# DATASCAPE 2026 — ResNet-34 Legacy Relabel + Crop, 5-Fold

Notebook ini hanya melatih dan melakukan inference. Seluruh label audit, near-deduplication, koreksi kualitas, dan YOLO crop harus sudah selesai di notebook Preprocessing V2.

Input:

- `processed_train_manifest.csv`;
- `processed_test_manifest.csv`;
- salinan global dan crop di folder preprocessing V2.

Output:

- checkpoint lima fold;
- OOF macro F1;
- ensemble global+crop probabilities;
- `submission_resnet34_processed_5fold.csv`.

In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


## AI Declaration dan batas model

OpenAI Codex digunakan untuk coding dan dokumentasi. ResNet-34 pada notebook ini memakai `weights=None`; tidak ada bobot atau data eksternal. Notebook ini tidak mengubah label dan tidak menjalankan YOLO.

Q&A panitia per 2 Oktober 2026 sebenarnya memperbolehkan pretrained model dan image encoder CLIP/VLM yang di-fine-tune pada data train resmi. Karena itu, `weights=None` di sini adalah pilihan baseline from-scratch, bukan kewajiban aturan.


In [ ]:
import importlib.util, json, math, os, random, subprocess, sys, time
from pathlib import Path

required = {'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow', 'matplotlib': 'matplotlib',
            'seaborn': 'seaborn', 'sklearn': 'scikit-learn', 'torch': 'torch',
            'torchvision': 'torchvision', 'tqdm': 'tqdm'}
missing = [p for m, p in required.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image, ImageOps
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules

## 1. Konfigurasi dan manifest final

In [ ]:
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: i for i, label in enumerate(CLASSES)}
INDEX_TO_CLASS = {i: label for label, i in CLASS_TO_INDEX.items()}

USE_GOOGLE_DRIVE = True
PREPROCESSED_DIR = Path('/content/drive/MyDrive/DATASCAPE/preprocessing_v2')
OUTPUT_DIR = Path('/content/drive/MyDrive/DATASCAPE/resnet34_processed_5fold')
IMAGE_SIZE = 320
BATCH_SIZE = 128  # fallback manual untuk GPU premium
AUTO_BATCH_SIZE = True
BATCH_CANDIDATES = (256, 192, 160, 128, 96, 64)
MAX_MEMORY_FRACTION = 0.90
MAX_EPOCHS = 30
PATIENCE = 7
REFERENCE_BATCH_SIZE = 32
REFERENCE_LR = 3e-4
BASE_LR = REFERENCE_LR
MIN_LR = 1e-6
WARMUP_EPOCHS = 2
WEIGHT_DECAY = 1e-4
BALANCE_STRENGTH = 0.5
NUM_WORKERS = 2
RESUME_FOLDS = True
GLOBAL_PROBABILITY_WEIGHT = 0.60
CROP_PROBABILITY_WEIGHT = 0.40

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_frame = pd.read_csv(PREPROCESSED_DIR / 'processed_train_manifest.csv')
test_frame = pd.read_csv(PREPROCESSED_DIR / 'processed_test_manifest.csv')
normalization = json.loads((PREPROCESSED_DIR / 'normalization.json').read_text())
RGB_MEAN, RGB_STD = tuple(normalization['mean']), tuple(normalization['std'])
for column in ['global_relpath', 'crop_relpath']:
    train_frame[column.replace('relpath', 'path')] = train_frame[column].map(lambda p: str(PREPROCESSED_DIR / p))
    test_frame[column.replace('relpath', 'path')] = test_frame[column].map(lambda p: str(PREPROCESSED_DIR / p))
assert train_frame.source_split.eq('train').all() and test_frame.source_split.eq('test').all()
assert set(train_frame.label) == set(CLASSES)
assert train_frame.groupby('pixel_sha256').size().max() == 1
for column in ['global_path', 'crop_path']:
    assert train_frame[column].map(lambda p: Path(p).is_file()).all()
    assert test_frame[column].map(lambda p: Path(p).is_file()).all()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.benchmark = True
display(pd.crosstab(train_frame.fold, train_frame.label).reindex(columns=CLASSES, fill_value=0))

## 2. Dataset global+crop dan ResNet-34

In [ ]:
class Letterbox:
    def __init__(self, size): self.size = int(size)
    def __call__(self, image):
        return ImageOps.pad(image, (self.size, self.size), Image.Resampling.BILINEAR,
                            color=tuple(int(value * 255) for value in RGB_MEAN))


train_transform = transforms.Compose([
    Letterbox(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05),
    transforms.ToTensor(), transforms.Normalize(RGB_MEAN, RGB_STD),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.07), value='random'),
])
eval_transform = transforms.Compose([
    Letterbox(IMAGE_SIZE),
    transforms.ToTensor(), transforms.Normalize(RGB_MEAN, RGB_STD),
])


class ProcessedDataset(Dataset):
    def __init__(self, frame, mode):
        self.frame = frame.reset_index(drop=True); self.mode = mode
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        if self.mode == 'train':
            path = row.crop_path if random.random() < 0.5 else row.global_path
            tensor = train_transform(Image.open(path).convert('RGB'))
            return tensor, CLASS_TO_INDEX[row.label], index
        global_tensor = eval_transform(Image.open(row.global_path).convert('RGB'))
        crop_tensor = eval_transform(Image.open(row.crop_path).convert('RGB'))
        views = torch.stack([global_tensor, crop_tensor])
        if self.mode == 'valid': return views, CLASS_TO_INDEX[row.label], index
        return views, index


def sampler(frame):
    counts = frame.label.value_counts()
    weights = frame.label.map(lambda label: counts[label] ** -BALANCE_STRENGTH).to_numpy(copy=True)
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), len(frame), replacement=True)


def loader(dataset, batch_size, sample=None):
    options = dict(dataset=dataset, batch_size=batch_size, sampler=sample, shuffle=False,
                   num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0: options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda': raise RuntimeError('Aktifkan GPU Colab.')


def build_model():
    model = models.resnet34(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.25), nn.Linear(model.fc.in_features, len(CLASSES)))
    return model.to(DEVICE, memory_format=torch.channels_last)


def probe_batch():
    if not AUTO_BATCH_SIZE: return BATCH_SIZE
    total = torch.cuda.get_device_properties(0).total_memory
    for batch in BATCH_CANDIDATES:
        model = optimizer = images = targets = loss = None
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model = build_model(); optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR)
            images = torch.randn(batch, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE).contiguous(memory_format=torch.channels_last)
            targets = torch.randint(0, len(CLASSES), (batch,), device=DEVICE)
            with torch.autocast('cuda', dtype=torch.float16): loss = nn.functional.cross_entropy(model(images), targets)
            loss.backward(); optimizer.step(); torch.cuda.synchronize()
            fraction = torch.cuda.max_memory_reserved() / total
            print(f'batch {batch}: {fraction:.1%} VRAM')
            if fraction <= MAX_MEMORY_FRACTION: return batch
        except torch.cuda.OutOfMemoryError:
            print(f'batch {batch}: OOM')
        finally:
            del model, optimizer, images, targets, loss; torch.cuda.empty_cache()
    raise RuntimeError('Tidak ada batch yang muat.')


BATCH_SIZE = probe_batch()
EVAL_BATCH_SIZE = max(8, BATCH_SIZE // 2)
BASE_LR = REFERENCE_LR * math.sqrt(BATCH_SIZE / REFERENCE_BATCH_SIZE)
print('Training batch:', BATCH_SIZE, '| evaluation batch:', EVAL_BATCH_SIZE, '| base LR:', BASE_LR)

## 3. Lima fold, cosine schedule, dan ensemble probabilitas

In [ ]:
def lr_at(epoch):
    if epoch < WARMUP_EPOCHS: return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(MAX_EPOCHS - WARMUP_EPOCHS - 1, 1)
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * min(progress, 1)))


def predict_views(model, data_loader, labeled):
    model.eval(); probabilities=[]; targets_all=[]; indices_all=[]
    weights = torch.tensor([GLOBAL_PROBABILITY_WEIGHT, CROP_PROBABILITY_WEIGHT], device=DEVICE).view(1, 2, 1)
    with torch.inference_mode():
        for batch in tqdm(data_loader, leave=False):
            if labeled:
                views, targets, indices = batch; targets_all.append(targets.numpy())
            else:
                views, indices = batch
            b, v, c, h, w = views.shape
            views = views.reshape(b * v, c, h, w).to(DEVICE, memory_format=torch.channels_last)
            with torch.autocast('cuda', dtype=torch.float16):
                probs = model(views).softmax(1).view(b, v, -1)
            probabilities.append((probs * weights).sum(1).float().cpu().numpy()); indices_all.append(indices.numpy())
    order = np.argsort(np.concatenate(indices_all))
    result = {'probabilities': np.concatenate(probabilities)[order]}
    if labeled: result['targets'] = np.concatenate(targets_all)[order]
    return result


def train_fold(fold):
    fold_dir = OUTPUT_DIR / f'fold_{fold}'; fold_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path, complete_path = fold_dir / 'best.pt', fold_dir / 'complete.json'
    train = train_frame[train_frame.fold.ne(fold)].reset_index(drop=True)
    valid = train_frame[train_frame.fold.eq(fold)].reset_index(drop=True)
    if RESUME_FOLDS and checkpoint_path.is_file() and complete_path.is_file():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model = build_model(); model.load_state_dict(checkpoint['model_state'])
        return model, valid, checkpoint
    train_loader = loader(ProcessedDataset(train, 'train'), BATCH_SIZE, sampler(train))
    valid_loader = loader(ProcessedDataset(valid, 'valid'), EVAL_BATCH_SIZE)
    model = build_model(); criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda'); best=-1.0; stale=0; history=[]
    for epoch in range(MAX_EPOCHS):
        learning_rate = lr_at(epoch)
        for group in optimizer.param_groups: group['lr'] = learning_rate
        model.train(); total_loss=0.0; total=0; started=time.time()
        for images, targets, _ in tqdm(train_loader, desc=f'fold {fold} epoch {epoch + 1}', leave=False):
            images = images.to(DEVICE, memory_format=torch.channels_last); targets = targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16): loss = criterion(model(images), targets)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += float(loss.item()) * len(images); total += len(images)
        result = predict_views(model, valid_loader, True)
        score = f1_score(result['targets'], result['probabilities'].argmax(1), average='macro', zero_division=0)
        row = {'epoch': epoch + 1, 'lr': learning_rate, 'train_loss': total_loss / total,
               'valid_macro_f1': score, 'seconds': time.time() - started}
        history.append(row); pd.DataFrame(history).to_csv(fold_dir / 'history.csv', index=False)
        print(f"fold {fold} epoch {epoch + 1}: loss={row['train_loss']:.4f} macro_f1={score:.4f}")
        if score > best:
            best=score; stale=0
            torch.save({'model_state': model.state_dict(), 'epoch': epoch + 1, 'macro_f1': score}, checkpoint_path)
        else:
            stale += 1
            if stale >= PATIENCE: break
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state']); complete_path.write_text(json.dumps(checkpoint | {'model_state': 'stored_in_pt'}, default=str))
    return model, valid, checkpoint


oof = np.full((len(train_frame), len(CLASSES)), np.nan, np.float32)
test_sum = np.zeros((len(test_frame), len(CLASSES)), np.float64)
test_loader = loader(ProcessedDataset(test_frame, 'test'), EVAL_BATCH_SIZE)
fold_rows=[]
for fold in range(5):
    model, valid, checkpoint = train_fold(fold)
    valid_result = predict_views(model, loader(ProcessedDataset(valid, 'valid'), EVAL_BATCH_SIZE), True)
    indices = train_frame.index[train_frame.fold.eq(fold)].to_numpy(); oof[indices] = valid_result['probabilities']
    test_result = predict_views(model, test_loader, False); test_sum += test_result['probabilities']
    fold_rows.append({'fold': fold, 'epoch': checkpoint['epoch'], 'macro_f1': checkpoint['macro_f1']})
    np.save(OUTPUT_DIR / f'fold_{fold}' / 'test_probabilities.npy', test_result['probabilities'])
    del model; torch.cuda.empty_cache()

assert np.isfinite(oof).all()
test_probabilities = (test_sum / 5).astype(np.float32)
np.save(OUTPUT_DIR / 'oof_probabilities.npy', oof); np.save(OUTPUT_DIR / 'test_probabilities.npy', test_probabilities)
pd.DataFrame(fold_rows).to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)

## 4. OOF evaluation dan submission

In [ ]:
targets = train_frame.label.map(CLASS_TO_INDEX).to_numpy(); predictions = oof.argmax(1)
macro_f1 = f1_score(targets, predictions, average='macro', zero_division=0)
report = pd.DataFrame(classification_report(targets, predictions, target_names=CLASSES,
                                            output_dict=True, zero_division=0)).T
report.to_csv(OUTPUT_DIR / 'oof_classification_report.csv'); display(report)
matrix = confusion_matrix(targets, predictions, normalize='true')
plt.figure(figsize=(6, 5)); sns.heatmap(matrix, annot=True, fmt='.3f', cmap='Blues',
                                       xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'OOF macro F1 = {macro_f1:.4f}'); plt.xlabel('Prediksi'); plt.ylabel('Label')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'oof_confusion_matrix.png', dpi=170); plt.show()

submission = pd.DataFrame({'filename': test_frame.filename.astype(str),
                           'label': [INDEX_TO_CLASS[i] for i in test_probabilities.argmax(1)]})
submission.to_csv(OUTPUT_DIR / 'submission_resnet34_processed_5fold.csv', index=False)
summary = {'model': 'ResNet34 weights=None', 'training_images': int(len(train_frame)),
           'oof_macro_f1': float(macro_f1), 'batch_size': BATCH_SIZE,
           'global_weight': GLOBAL_PROBABILITY_WEIGHT, 'crop_weight': CROP_PROBABILITY_WEIGHT,
           'external_weights': False}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2)); print('Submission:', OUTPUT_DIR / 'submission_resnet34_processed_5fold.csv')